# 26 — End-to-End Adaptive Road Vision Pipeline

This notebook integrates the available project components into one inference pipeline:

**Input image → condition classification → optional enhancement → object detection/tracking → optional relative depth → rule-based risk score → structured outputs**

### Important limitations
- This notebook performs inference; it does not train missing models.
- It requires a trained condition-classifier checkpoint and the selected YOLO detector. It will stop with an actionable error if either is missing rather than silently using random weights or a different detector.
- Enhancement and depth are optional. Missing enhancement checkpoints disable that enhancement; depth import/model errors disable depth with a warning.
- ByteTrack IDs and persistence are meaningful only when inputs are ordered frames from the same continuous video/sequence. A folder of unrelated still images is not valid tracking evaluation.
- Monocular depth is relative, not metric distance unless separately calibrated.
- The reported risk score is a heuristic perception score, not collision probability or a safety certification.


In [ ]:
from pathlib import Path
import os, json, time, warnings, math, re
from collections import defaultdict

warnings.filterwarnings("ignore")

# Update this path if your project is stored elsewhere.
PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
if not PROJECT_ROOT.exists():
    # Useful fallback when the notebook is run from inside the project directory.
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next((p for p in candidates if (p / "models").exists()), PROJECT_ROOT)
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}. Edit PROJECT_ROOT in this cell to your Adaptive_Road_Vision folder."
    )

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results" / "pipeline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Pipeline results:", RESULTS_DIR)


## 1. Configuration

For a first run, keep `MAX_FRAMES` small. Set it to `None` to process the full test set.

The pipeline can also run on a custom image folder by changing `INPUT_MODE` and `CUSTOM_IMAGE_DIR`.

In [ ]:
INPUT_MODE = "test"          # "test" or "custom"
MAX_FRAMES = 100             # None = all available images
IMAGE_SIZE = 640
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50
TRACKER_CFG = "bytetrack.yaml"

# Depth is optional. Set False to avoid Transformers/depth-model dependencies.
USE_DEPTH = True
DEPTH_MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
DEPTH_SCALE_M_PER_UNIT = None   # leave None unless you have a valid calibration

CONDITION_NAMES = ["Day", "Night", "Rain", "Fog", "Unknown"]
SAVE_VISUALIZATIONS = True
VIS_MAX = 20

# Dataset layout candidates are checked in order.
TEST_IMAGE_DIR_CANDIDATES = [
    PROJECT_ROOT / "datasets" / "final" / "images" / "test",
    PROJECT_ROOT / "datasets" / "final_condition_balanced_v2" / "images" / "test",
    PROJECT_ROOT / "datasets" / "final_condition_balanced_v2" / "test" / "images",
]
CUSTOM_IMAGE_DIR = PROJECT_ROOT / "datasets" / "custom"
TEST_IMAGE_DIR = next((p for p in TEST_IMAGE_DIR_CANDIDATES if p.exists()), TEST_IMAGE_DIR_CANDIDATES[0])

print("Input mode:", INPUT_MODE)
print("MAX_FRAMES:", MAX_FRAMES)
print("Test image directory:", TEST_IMAGE_DIR)


## 2. Locate required model artifacts

In [ ]:
def first_existing(candidates):
    return next((Path(p) for p in candidates if Path(p).is_file()), None)

# Only look for explicitly selected-detector checkpoint names; do not silently substitute another detector.
condition_candidates = [
    MODELS_DIR / "condition" / "condition_resnet18_best.pth",
    MODELS_DIR / "condition" / "best_model.pth",
    MODELS_DIR / "condition" / "best.pth",
]
detector_candidates = [
    MODELS_DIR / "detection" / "final_selected_detector.pt",
    MODELS_DIR / "detection" / "selected_detector.pt",
    MODELS_DIR / "detection" / "final_best.pt",
]
required = {
    "condition": first_existing(condition_candidates),
    "night": first_existing([MODELS_DIR / "enhancement" / "night" / "night_enhancer_best.pth", MODELS_DIR / "enhancement" / "night" / "best.pth"]),
    "rain": first_existing([MODELS_DIR / "enhancement" / "rain" / "rain_enhancer_best.pth", MODELS_DIR / "enhancement" / "rain" / "best.pth"]),
    "fog": first_existing([MODELS_DIR / "enhancement" / "fog" / "fog_enhancer_best.pth", MODELS_DIR / "enhancement" / "fog" / "best.pth"]),
    "detector": first_existing(detector_candidates),
}
for name, path in required.items():
    print(f"{name:10s}: {'FOUND' if path else 'MISSING'} -> {path if path else 'No matching checkpoint found'}")

missing = [name for name in ("condition", "detector") if required[name] is None]
if missing:
    raise FileNotFoundError(
        "Required trained model(s) missing: " + ", ".join(missing) + ". "
        "Train the corresponding notebook(s) and place the checkpoint in the models folder, "
        "or update the candidate paths in this cell. No untrained-model fallback is used."
    )


## 3. Imports and device

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

## 4. Model definitions

The enhancement network below matches the compact residual encoder-decoder interface used by the Night/Rain/Fog training notebooks. Checkpoint loading is strict so an architecture mismatch is surfaced instead of silently producing invalid inference.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
        )
    def forward(self, x):
        return x + self.block(x)

class CompactResidualEnhancer(nn.Module):
    def __init__(self, base=32, blocks=4):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(3, base, 3, padding=1), nn.ReLU(inplace=True))
        self.down1 = nn.Sequential(nn.Conv2d(base, base*2, 4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.res = nn.Sequential(*[ResidualBlock(base*2) for _ in range(blocks)])
        self.up1 = nn.Sequential(nn.ConvTranspose2d(base*2, base, 4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.tail = nn.Conv2d(base, 3, 3, padding=1)
    def forward(self, x):
        y = self.head(x)
        y = self.down1(y)
        y = self.res(y)
        y = self.up1(y)
        delta = torch.tanh(self.tail(y))
        if y.shape[-2:] != x.shape[-2:]:
            y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
            delta = torch.tanh(self.tail(y))
        return torch.clamp(x + 0.5 * delta, 0, 1)

def load_checkpoint_state(model, path):
    # weights_only is supported in recent PyTorch; fall back for older releases.
    try:
        ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        ckpt = torch.load(path, map_location=DEVICE)
    if isinstance(ckpt, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in ckpt and isinstance(ckpt[key], dict):
                ckpt = ckpt[key]
                break
    if not isinstance(ckpt, dict):
        raise ValueError(f"Unsupported checkpoint format: {path}")
    # Strip common DataParallel prefix.
    state = {k.removeprefix("module."): v for k, v in ckpt.items()}
    model.load_state_dict(state, strict=True)
    return model

print("Enhancer architecture and checkpoint loader ready.")


## 5. Load condition classifier

In [ ]:
condition_model = models.resnet18(weights=None)
condition_model.fc = nn.Linear(condition_model.fc.in_features, len(CONDITION_NAMES))
condition_model = load_checkpoint_state(condition_model, required["condition"])
condition_model.to(DEVICE).eval()
print("Condition classifier loaded from:", required["condition"])

condition_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


## 6. Load condition-specific enhancement models

In [ ]:
enhancers = {}
for cond in ["Night", "Rain", "Fog"]:
    key = cond.lower()
    path = required[key]
    if path is not None and Path(path).is_file():
        model = CompactResidualEnhancer().to(DEVICE)
        try:
            load_checkpoint_state(model, path)
            model.eval()
            enhancers[cond] = model
            print(f"{cond} enhancer loaded.")
        except (RuntimeError, ValueError, KeyError) as e:
            print(f"WARNING: {cond} enhancer checkpoint could not be loaded; enhancement disabled.")
            print("         Error:", str(e).splitlines()[0])
    else:
        print(f"{cond} enhancer missing; enhancement disabled.")

enhance_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])


## 7. Load detector

In [ ]:
try:
    from ultralytics import YOLO
except ImportError as e:
    raise ImportError("Ultralytics is required. Install it in this notebook's environment with: pip install ultralytics") from e

if required["detector"] is None or not Path(required["detector"]).is_file():
    raise FileNotFoundError("Selected detector checkpoint is missing. Check the checkpoint paths in section 2.")
detector = YOLO(str(required["detector"]))
print("Selected detector loaded:", required["detector"])


## 8. Optional Depth Anything V2 loader

In [ ]:
depth_processor = None
depth_model = None
depth_error = None

if USE_DEPTH:
    try:
        from transformers import AutoImageProcessor, AutoModelForDepthEstimation
        depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_ID)
        depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_ID).to(DEVICE).eval()
        print("Depth model loaded:", DEPTH_MODEL_ID)
    except Exception as e:
        depth_error = f"{type(e).__name__}: {e}"
        depth_processor, depth_model = None, None
        USE_DEPTH = False
        print("WARNING: Depth disabled because the model could not be loaded.")
        print("Reason:", depth_error)
        print("You can continue without depth, or fix the Transformers/PyTorch environment and rerun this cell.")
else:
    print("Depth disabled by configuration.")


## 9. Risk configuration

In [ ]:
CLASS_WEIGHTS = {
    0: 1.30,  # pedestrian
    1: 1.25,  # rider
    2: 1.00,  # car
    3: 1.05,  # truck
    4: 1.05,  # bus
    5: 1.00,  # train
    6: 1.20,  # motorcycle
    7: 1.20,  # bicycle
}
CONDITION_FACTORS = {
    "Day": 1.00,
    "Night": 1.20,
    "Rain": 1.15,
    "Fog": 1.25,
    "Unknown": 1.05,
}
RISK_THRESHOLDS = {
    "LOW": 0.25,
    "MEDIUM": 0.50,
    "HIGH": 0.75,
}

def risk_level(score):
    if score < RISK_THRESHOLDS["LOW"]:
        return "VERY_LOW"
    if score < RISK_THRESHOLDS["MEDIUM"]:
        return "LOW"
    if score < RISK_THRESHOLDS["HIGH"]:
        return "MEDIUM"
    return "HIGH"

def proximity_from_depth(depth_value):
    if depth_value is None or not np.isfinite(depth_value):
        return 0.0
    # Depth Anything's predicted_depth is an inverse-depth-like relative map:
    # larger normalized values generally indicate closer regions. This is NOT metres.
    d = float(np.clip(depth_value, 0.0, 1.0))
    return d

def object_risk(depth_value, confidence, persistence, class_id):
    proximity = proximity_from_depth(depth_value)
    confidence_factor = float(np.clip(confidence, 0.0, 1.0))
    persistence_factor = float(np.clip(persistence, 0.0, 1.0))
    class_weight = CLASS_WEIGHTS.get(int(class_id), 1.0)
    return proximity * confidence_factor * (0.5 + 0.5*persistence_factor) * class_weight

def aggregate_frame_risk(object_scores, condition):
    base = 1.0 - math.exp(-float(np.sum(object_scores))) if len(object_scores) else 0.0
    return float(np.clip(base * CONDITION_FACTORS.get(condition, 1.05), 0.0, 1.0))

## 10. Input image discovery

In [ ]:
def discover_images():
    root = TEST_IMAGE_DIR if INPUT_MODE == "test" else CUSTOM_IMAGE_DIR
    if INPUT_MODE not in {"test", "custom"}:
        raise ValueError("INPUT_MODE must be 'test' or 'custom'.")
    if not root.exists():
        raise FileNotFoundError(f"Input image directory does not exist: {root}. Update the path in section 1.")
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts)
    if MAX_FRAMES is not None:
        files = files[:max(0, int(MAX_FRAMES))]
    return files

image_paths = discover_images()
print("Images selected:", len(image_paths))
if not image_paths:
    raise FileNotFoundError(f"No input images found under {TEST_IMAGE_DIR if INPUT_MODE == 'test' else CUSTOM_IMAGE_DIR}.")
if INPUT_MODE == "test":
    print("NOTE: If these are independent test stills, ByteTrack persistence across them is not a valid temporal tracking metric.")


## 11. Helper functions

In [ ]:
@torch.inference_mode()
def predict_condition(pil_img):
    x = condition_tf(pil_img).unsqueeze(0).to(DEVICE)
    logits = condition_model(x)
    probs = torch.softmax(logits, dim=1)[0]
    idx = int(torch.argmax(probs).item())
    return CONDITION_NAMES[idx], float(probs[idx].item()), probs.detach().cpu().numpy()

@torch.inference_mode()
def enhance_image(pil_img, condition):
    if condition not in enhancers:
        return pil_img, False
    x = enhance_tf(pil_img).unsqueeze(0).to(DEVICE)
    y = enhancers[condition](x)[0].detach().cpu().permute(1, 2, 0).numpy()
    y = (np.clip(y, 0, 1) * 255).astype(np.uint8)
    # Resize back to original dimensions because the enhancement transform may resize to 256x256.
    out = Image.fromarray(y).resize(pil_img.size, Image.Resampling.BILINEAR)
    return out, True

@torch.inference_mode()
def estimate_depth_map(pil_img):
    if not USE_DEPTH or depth_model is None or depth_processor is None:
        return None
    inputs = depth_processor(images=pil_img, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    outputs = depth_model(**inputs)
    predicted_depth = outputs.predicted_depth
    prediction = F.interpolate(predicted_depth.unsqueeze(1), size=(pil_img.height, pil_img.width), mode="bicubic", align_corners=False).squeeze(0).squeeze(0)
    arr = prediction.float().cpu().numpy()
    if arr.size == 0 or not np.isfinite(arr).any():
        return None
    finite = arr[np.isfinite(arr)]
    lo, hi = np.percentile(finite, 2), np.percentile(finite, 98)
    if not np.isfinite(lo + hi) or hi - lo < 1e-8:
        return np.full(arr.shape, 0.5, dtype=np.float32)
    norm = (arr - lo) / (hi - lo)
    return np.clip(np.nan_to_num(norm, nan=0.5, posinf=1.0, neginf=0.0), 0, 1).astype(np.float32)

def box_depth(depth_map, xyxy):
    if depth_map is None:
        return None
    x1, y1, x2, y2 = [int(round(v)) for v in xyxy]
    h, w = depth_map.shape[:2]
    x1, x2 = max(0, min(w, x1)), max(0, min(w, x2))
    y1, y2 = max(0, min(h, y1)), max(0, min(h, y2))
    if x2 <= x1 or y2 <= y1:
        return None
    ix, iy = int(0.15 * (x2-x1)), int(0.15 * (y2-y1))
    crop = depth_map[y1+iy:max(y1+iy+1, y2-iy), x1+ix:max(x1+ix+1, x2-ix)]
    if crop.size == 0:
        crop = depth_map[y1:y2, x1:x2]
    return float(np.nanmedian(crop)) if crop.size and np.isfinite(crop).any() else None

def sequence_key(path):
    stem = path.stem
    m = re.match(r"(.+?)[-_](\d+)$", stem)
    return m.group(1) if m else stem

def persistence_from_track(track_counts, track_id):
    if track_id is None:
        return 0.0
    return float(np.clip(track_counts.get(int(track_id), 0) / 10.0, 0, 1))

def safe_float(v):
    try:
        value = float(v)
        return value if np.isfinite(value) else None
    except (TypeError, ValueError):
        return None


## 12. Run the integrated pipeline

In [ ]:
per_frame_rows = []
object_rows = []
visual_paths = []

track_counts = defaultdict(int)
sequence_state = {}
current_sequence = None

start_all = time.perf_counter()

for frame_idx, image_path in enumerate(image_paths):
    t0 = time.perf_counter()
    seq_id = sequence_key(image_path)
    # Do not carry tracker IDs or persistence counts across detected sequence boundaries.
    if seq_id != current_sequence:
        if current_sequence is not None:
            try:
                detector.predictor = None  # force Ultralytics to reinitialize tracking state
            except Exception:
                pass
        track_counts.clear()
        current_sequence = seq_id

    try:
        original = Image.open(image_path).convert("RGB")
    except Exception as e:
        print("Skipping unreadable image:", image_path, e)
        continue

    condition, condition_conf, condition_probs = predict_condition(original)
    enhanced, enhancement_applied = enhance_image(original, condition)

    # ByteTrack inference on the selected detector.
    results = detector.track(
        source=np.asarray(enhanced),
        persist=True,
        tracker=TRACKER_CFG,
        imgsz=IMAGE_SIZE,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        verbose=False,
    )
    result = results[0]

    depth_map = estimate_depth_map(enhanced) if USE_DEPTH else None

    detections = []
    if result.boxes is not None and len(result.boxes) > 0:
        xyxy = result.boxes.xyxy.detach().cpu().numpy()
        confs = result.boxes.conf.detach().cpu().numpy()
        clss = result.boxes.cls.detach().cpu().numpy().astype(int)
        ids = (
            result.boxes.id.detach().cpu().numpy().astype(int)
            if result.boxes.id is not None else
            np.full(len(xyxy), -1, dtype=int)
        )

        for box, conf, cls_id, track_id in zip(xyxy, confs, clss, ids):
            tid = None if int(track_id) < 0 else int(track_id)
            if tid is not None:
                track_counts[tid] += 1

            d = box_depth(depth_map, box)
            persistence = persistence_from_track(track_counts, tid)
            r = object_risk(d, float(conf), persistence, int(cls_id))

            detections.append({
                "class_id": int(cls_id),
                "confidence": float(conf),
                "track_id": tid,
                "xyxy": [float(v) for v in box],
                "relative_depth": d,
                "persistence": persistence,
                "object_risk": r,
            })

            object_rows.append({
                "frame_index": frame_idx,
                "image_name": image_path.name,
                "sequence_id": seq_id,
                "condition": condition,
                "condition_confidence": condition_conf,
                "enhancement_applied": enhancement_applied,
                **{f"condition_prob_{c}": float(condition_probs[i]) for i,c in enumerate(CONDITION_NAMES)},
                **detections[-1],
            })

    obj_scores = [d["object_risk"] for d in detections]
    frame_risk = aggregate_frame_risk(obj_scores, condition)

    elapsed = time.perf_counter() - t0
    per_frame_rows.append({
        "frame_index": frame_idx,
        "image_name": image_path.name,
        "image_path": str(image_path),
        "sequence_id": seq_id,
        "condition": condition,
        "condition_confidence": condition_conf,
        "enhancement_applied": enhancement_applied,
        "num_detections": len(detections),
        "num_active_tracks": len({d["track_id"] for d in detections if d["track_id"] is not None}),
        "mean_detection_confidence": float(np.mean([d["confidence"] for d in detections])) if detections else 0.0,
        "frame_risk": frame_risk,
        "risk_level": risk_level(frame_risk),
        "depth_available": depth_map is not None,
        "runtime_seconds": elapsed,
        "measured_fps": 1.0 / elapsed if elapsed > 0 else np.nan,
    })

    # Lightweight visualization for a limited number of frames.
    if SAVE_VISUALIZATIONS and frame_idx < VIS_MAX:
        try:
            plotted = result.plot()
            # Add a simple pipeline banner.
            img = Image.fromarray(plotted[..., ::-1]) if plotted.shape[-1] == 3 else Image.fromarray(plotted)
            out_path = RESULTS_DIR / "visualizations" / f"{frame_idx:05d}_{image_path.stem}.jpg"
            out_path.parent.mkdir(parents=True, exist_ok=True)
            img.save(out_path, quality=90)
            visual_paths.append(str(out_path))
        except Exception as e:
            print("Visualization warning:", e)

    if (frame_idx + 1) % 10 == 0 or frame_idx == 0:
        print(
            f"[{frame_idx+1:>4}/{len(image_paths)}] "
            f"{condition:>7} | enh={enhancement_applied} | "
            f"dets={len(detections):>2} | risk={frame_risk:.3f} | "
            f"{elapsed:.3f}s"
        )

total_elapsed = time.perf_counter() - start_all
print("\nPipeline finished.")
print("Frames processed:", len(per_frame_rows))
print("Total time:", round(total_elapsed, 3), "s")
print("Measured throughput:", round(len(per_frame_rows)/total_elapsed, 3), "FPS" if total_elapsed else "")

## 13. Save structured outputs

In [ ]:
frames_df = pd.DataFrame(per_frame_rows)
objects_df = pd.DataFrame(object_rows)

frames_csv = RESULTS_DIR / "pipeline_per_frame.csv"
objects_csv = RESULTS_DIR / "pipeline_object_details.csv"
frames_df.to_csv(frames_csv, index=False)
objects_df.to_csv(objects_csv, index=False)

if frames_df.empty:
    condition_summary = pd.DataFrame(columns=["condition", "frames", "mean_condition_confidence", "enhancement_rate", "mean_detections", "mean_active_tracks", "mean_frame_risk", "high_risk_rate", "mean_fps"])
    risk_summary = pd.DataFrame(columns=["risk_level", "frames", "fraction"])
else:
    condition_summary = frames_df.groupby("condition", dropna=False).agg(
        frames=("frame_index", "count"),
        mean_condition_confidence=("condition_confidence", "mean"),
        enhancement_rate=("enhancement_applied", "mean"),
        mean_detections=("num_detections", "mean"),
        mean_active_tracks=("num_active_tracks", "mean"),
        mean_frame_risk=("frame_risk", "mean"),
        high_risk_rate=("risk_level", lambda s: float(np.mean(s == "HIGH"))),
        mean_fps=("measured_fps", "mean"),
    ).reset_index()
    risk_summary = frames_df["risk_level"].value_counts(dropna=False).rename_axis("risk_level").reset_index(name="frames")
    risk_summary["fraction"] = risk_summary["frames"] / max(len(frames_df), 1)

condition_csv = RESULTS_DIR / "pipeline_condition_summary.csv"
risk_csv = RESULTS_DIR / "pipeline_risk_summary.csv"
condition_summary.to_csv(condition_csv, index=False)
risk_summary.to_csv(risk_csv, index=False)

print("Saved:", frames_csv)
print("Saved:", objects_csv)
print("Saved:", condition_csv)
print("Saved:", risk_csv)


## 14. Pipeline report

In [ ]:
report = {
    "project_root": str(PROJECT_ROOT),
    "input_mode": INPUT_MODE,
    "input_image_count": len(image_paths),
    "processed_frame_count": int(len(frames_df)),
    "total_runtime_seconds": float(total_elapsed),
    "measured_pipeline_fps": float(len(frames_df) / total_elapsed) if total_elapsed and len(frames_df) else None,
    "device": str(DEVICE),
    "cuda_available": bool(torch.cuda.is_available()),
    "selected_detector": str(required["detector"]),
    "condition_model": str(required["condition"]),
    "enhancement_models": {k: (str(required[k.lower()]) if required[k.lower()] else None) for k in ["Night", "Rain", "Fog"]},
    "depth_enabled": bool(USE_DEPTH),
    "depth_model_id": DEPTH_MODEL_ID if USE_DEPTH else None,
    "depth_error": depth_error,
    "depth_scale_m_per_unit": DEPTH_SCALE_M_PER_UNIT,
    "tracker": TRACKER_CFG,
    "confidence_threshold": CONF_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "visualizations": visual_paths,
    "artifacts": {
        "per_frame_csv": str(frames_csv),
        "object_details_csv": str(objects_csv),
        "condition_summary_csv": str(condition_csv),
        "risk_summary_csv": str(risk_csv),
    },
    "notes": [
        "Condition classification, enhancement, detection, tracking, depth, and risk are integrated sequentially.",
        "Night/Rain/Fog enhancement runs only when a matching enhancer checkpoint was loaded.",
        "ByteTrack IDs are operational IDs; without continuous ordered video frames they are not valid tracking evaluation.",
        "Depth is relative unless a separate calibration scale is supplied.",
        "Measured FPS is for this notebook configuration and hardware; it is not a real-time performance guarantee.",
        "Risk score is a project-specific heuristic, not collision probability or a safety certification.",
    ],
}
report_path = RESULTS_DIR / "end_to_end_pipeline_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("Report:", report_path)


## 15. Quick inspection

In [ ]:
display(frames_df.head(10))

print("\nCondition summary:")
display(condition_summary)

print("\nRisk distribution:")
display(risk_summary)

if not objects_df.empty:
    print("\nObject details:")
    display(objects_df.head(10))
else:
    print("\nNo detections were recorded in the processed frames.")

## 16. Interpretation

This notebook demonstrates system integration; it does not independently validate every component.

For the final report, use the matching evaluation notebooks in your project and verify their numbering before citing them. Do not report tracking metrics from unrelated still images as temporal tracking performance. Report depth as relative unless metric calibration is available; interpret the depth direction according to the loaded model. Describe the risk output as a rule-based perception score, not as a collision probability or certified safety system.
